# StyleGAN3 Navier–Stokes Forecasting Baseline

This notebook assembles a self-contained training and evaluation pipeline that adapts the original StyleGAN3 implementation to the Navier–Stokes fixed-sparsity forecasting task.


In [ ]:
import glob
import json
import math
import os
import time
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader
from tqdm.auto import tqdm

import dnnlib
from training import training_loop
from training import dataset as sg_dataset_base
import legacy

from mmap_ninja import RaggedMmap


In [ ]:
CONFIG = {
    "device": "cuda:0" if torch.cuda.is_available() else "cpu",
    "mean": 0.0,
    "std": 2.4036,
    "clip_value": 5.0,
    "sparsity": 0.2,
    "mask_seed": 0,
    "train_pairs": 49000,
    "val_pairs": 4900,
    "data": {
        "base_dir": os.environ.get("NAVIERSTOKES_DATA_DIR", "/pscratch/sd/d/dpark1/NSData"),
        "dataset_name": os.environ.get("NAVIERSTOKES_DATASET", "100k"),
    },
    "gan": {
        "cfg": "stylegan3-t",
        "gamma": 0.2,
        "batch_size": 32,
        "batch_gpu": None,
        "total_kimg": 5000,
        "tick_kimg": 4,
        "snapshot_ticks": 10,
        "metrics": [],
        "augment": "ada",
        "augment_p": 0.0,
        "ada_target": 0.6,
        "ada_interval": 4,
        "resume_pkl": None,
        "freezed": 0,
        "map_depth": None,
        "cbase": 32768,
        "cmax": 512,
        "glr": None,
        "dlr": 0.002,
        "mbstd": 4,
        "seed": 0,
        "num_workers": 3,
        "truncation_psi": 1.0,
    },
    "evaluation": {
        "batch_size": 128,
        "num_workers": 2,
        "crps_k": 10,
    },
}

MEMMAP_PATH = os.path.join(CONFIG["data"]["base_dir"], CONFIG["data"]["dataset_name"])
if not os.path.exists(MEMMAP_PATH):
    raise FileNotFoundError(
        f"Expected Navier–Stokes memmap directory at {MEMMAP_PATH}. Update CONFIG['data'] before running."
    )


In [ ]:
VAL_SEED = 12345
TRAIN_SEED = 0


def dataset_tag_from_pairs(num_pairs: int) -> str:
    if num_pairs == 49000:
        return "full"
    if num_pairs == 4900:
        return "10pct"
    return f"{num_pairs}pairs"


def load_index_pairs(index_pairs=None, pairs_path: str = None) -> np.ndarray:
    if pairs_path is not None:
        data = np.load(pairs_path)
        if isinstance(data, np.lib.npyio.NpzFile):
            if "index_pairs" not in data:
                raise KeyError("NPZ file must contain an 'index_pairs' array")
            arr = data["index_pairs"]
        else:
            arr = data
        return np.asarray(arr, dtype=np.int64)
    if index_pairs is None:
        raise ValueError("Either index_pairs or pairs_path must be provided.")
    return np.asarray(index_pairs, dtype=np.int64)


def build_fixed_item_splits(
    mem: RaggedMmap,
    target_train_pairs: int,
    val_pairs: int = 4900,
    train_seed: int = TRAIN_SEED,
    val_seed: int = VAL_SEED,
) -> Tuple[np.ndarray, np.ndarray, int, np.ndarray, np.ndarray]:
    num_items_total = len(mem)
    if num_items_total == 0:
        raise ValueError("Empty Navier–Stokes memmap.")
    T = int(mem[0].shape[0])
    if T < 2:
        raise ValueError("Each sequence must provide at least two timesteps.")
    pairs_per_item = T - 1

    items_for_val = int(np.ceil(val_pairs / pairs_per_item))
    items_for_train = int(np.ceil(target_train_pairs / pairs_per_item))

    rng_val = np.random.default_rng(val_seed)
    val_items = np.sort(rng_val.choice(num_items_total, size=items_for_val, replace=False))
    remaining = np.setdiff1d(np.arange(num_items_total), val_items, assume_unique=True)
    if len(remaining) < items_for_train:
        raise RuntimeError("Insufficient items to satisfy requested train split size.")

    rng_train = np.random.default_rng(train_seed)
    train_items = np.sort(rng_train.choice(remaining, size=items_for_train, replace=False))

    def pairs_from_items(items: np.ndarray) -> np.ndarray:
        return np.asarray([(int(it), int(t)) for it in items for t in range(T - 1)], dtype=np.int64)

    train_pairs = pairs_from_items(train_items)[:target_train_pairs]
    val_pairs_arr = pairs_from_items(val_items)[:val_pairs]

    if set(map(tuple, train_pairs)).intersection(set(map(tuple, val_pairs_arr))):
        raise RuntimeError("Train and validation pairs overlap; check split configuration.")

    return train_pairs, val_pairs_arr, T, train_items, val_items


def compute_sample_id(item_idx: int, T: int, t: int) -> int:
    return int(item_idx) * T + int(t)


def make_fixed_mask(item_idx: int, t: int, T: int, sparsity: float, mask_seed: int, shape: Tuple[int, int]) -> np.ndarray:
    sid = compute_sample_id(item_idx, T, t)
    rng = np.random.default_rng(mask_seed + sid)
    return (rng.random(shape, dtype=np.float32) < sparsity).astype(np.float32)


def to_scaled(field: np.ndarray, mean: float, std: float, clip_value: float) -> np.ndarray:
    norm = (field - mean) / std
    clipped = np.clip(norm, -clip_value, clip_value)
    return clipped / clip_value


def scaled_to_uint8(scaled: np.ndarray) -> np.ndarray:
    scaled = np.clip(scaled, -1.0, 1.0)
    return np.rint((scaled + 1.0) * 0.5 * 255.0).astype(np.uint8)


def uint8_to_scaled(image_uint8: np.ndarray) -> np.ndarray:
    return image_uint8.astype(np.float32) / 255.0 * 2.0 - 1.0


def scaled_to_physical(scaled: np.ndarray, clip_value: float, mean: float, std: float) -> np.ndarray:
    return (np.clip(scaled, -1.0, 1.0) * clip_value * std) + mean


def build_condition_vector_from_field(
    field: np.ndarray,
    mask: np.ndarray,
    mean: float,
    std: float,
    clip_value: float,
) -> np.ndarray:
    scaled = to_scaled(field, mean, std, clip_value)
    sparse_scaled = scaled * mask
    return np.concatenate([sparse_scaled.reshape(-1), mask.reshape(-1)]).astype(np.float32)


def parse_condition_vector(vec: np.ndarray, height: int, width: int) -> Tuple[np.ndarray, np.ndarray]:
    split = height * width
    sparse = vec[:split].reshape(height, width)
    mask = vec[split:].reshape(height, width)
    return sparse, mask


def prepare_condition_tensor(x_norm: torch.Tensor, mask: torch.Tensor, clip_value: float) -> torch.Tensor:
    if mask.ndim == 3:
        mask = mask.unsqueeze(1)
    mask_f = mask.to(dtype=x_norm.dtype)
    scaled = torch.clamp(x_norm, -clip_value, clip_value) / clip_value
    sparse_scaled = scaled * mask_f
    return torch.cat([sparse_scaled.flatten(1), mask_f.flatten(1)], dim=1)


def scaled_to_physical_tensor(tensor: torch.Tensor, clip_value: float, mean: float, std: float) -> torch.Tensor:
    return tensor.clamp(-1.0, 1.0) * clip_value * std + mean


def compute_crps(preds: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    term1 = (preds - target.unsqueeze(0)).abs().mean(dim=0)
    diffs = (preds.unsqueeze(0) - preds.unsqueeze(1)).abs()
    term2 = 0.5 * diffs.mean(dim=(0, 1))
    return (term1 - term2).mean()


def export_training_config(config: dnnlib.EasyDict, destination: str) -> None:
    def _convert(obj):
        if isinstance(obj, dnnlib.EasyDict):
            return {k: _convert(v) for k, v in obj.items()}
        if isinstance(obj, dict):
            return {k: _convert(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [_convert(v) for v in obj]
        if isinstance(obj, (int, float, str, bool)) or obj is None:
            return obj
        return str(obj)

    with open(destination, "w", encoding="utf-8") as f:
        json.dump(_convert(config), f, indent=2)


In [ ]:
class NavierStokesStyleGANDataset(sg_dataset_base.Dataset):
    def __init__(
        self,
        memmap_path: str,
        index_pairs=None,
        pairs_path: str = None,
        sparsity: float = 0.2,
        mean: float = 0.0,
        std: float = 1.0,
        clip_value: float = 5.0,
        mask_seed: int = 0,
        name: str = "NavierStokesStyleGAN3",
        random_seed: int = 0,
        xflip: bool = False,
    ) -> None:
        self.memmap_path = memmap_path
        self.mem = RaggedMmap(memmap_path, mode="r")
        self.index_pairs = load_index_pairs(index_pairs=index_pairs, pairs_path=pairs_path)
        if len(self.index_pairs) == 0:
            raise ValueError("NavierStokesStyleGANDataset requires at least one (item, t) pair.")
        first_item = int(self.index_pairs[0, 0])
        seq0 = self.mem[first_item]
        self.T = int(seq0.shape[0])
        self.H = int(seq0.shape[1])
        self.W = int(seq0.shape[2])
        raw_shape = [len(self.index_pairs), 1, self.H, self.W]

        self.mean = float(mean)
        self.std = float(std)
        self.clip_value = float(clip_value)
        self.sparsity = float(sparsity)
        self.mask_seed = int(mask_seed)

        super().__init__(
            name=name,
            raw_shape=raw_shape,
            max_size=None,
            use_labels=True,
            xflip=xflip,
            random_seed=random_seed,
        )
        self._label_shape = [self.H * self.W * 2]

    def _load_raw_image(self, raw_idx: int) -> np.ndarray:
        item_idx, t = map(int, self.index_pairs[raw_idx])
        seq = self.mem[item_idx]
        target = np.asarray(seq[t + 1], dtype=np.float32)
        scaled = to_scaled(target, self.mean, self.std, self.clip_value)
        return scaled_to_uint8(scaled)[None, ...]

    def _build_label(self, raw_idx: int) -> np.ndarray:
        item_idx, t = map(int, self.index_pairs[raw_idx])
        seq = self.mem[item_idx]
        field = np.asarray(seq[t], dtype=np.float32)
        mask = make_fixed_mask(item_idx, t, self.T, self.sparsity, self.mask_seed, (self.H, self.W))
        return build_condition_vector_from_field(field, mask, self.mean, self.std, self.clip_value)

    def _flip_label(self, label: np.ndarray) -> np.ndarray:
        sparse, mask = parse_condition_vector(label, self.H, self.W)
        sparse = sparse[:, ::-1]
        mask = mask[:, ::-1]
        return np.concatenate([sparse.reshape(-1), mask.reshape(-1)]).astype(np.float32)

    def __getitem__(self, idx: int):
        raw_idx = int(self._raw_idx[idx])
        image = self._load_raw_image(raw_idx)
        if self._xflip[idx]:
            image = image[:, :, ::-1]
        label = self._build_label(raw_idx)
        if self._xflip[idx]:
            label = self._flip_label(label)
        return image.copy(), label.copy()

    def get_label(self, idx: int):
        raw_idx = int(self._raw_idx[idx])
        label = self._build_label(raw_idx)
        if self._xflip[idx]:
            label = self._flip_label(label)
        return label.copy()


class NavierStokesForecastingDataset(TorchDataset):
    def __init__(
        self,
        memmap_path: str,
        index_pairs=None,
        pairs_path: str = None,
        sparsity: float = 0.2,
        mean: float = 0.0,
        std: float = 1.0,
        clip_value: float = 5.0,
        mask_seed: int = 0,
        normalize: bool = True,
    ) -> None:
        self.memmap_path = memmap_path
        self.mem = RaggedMmap(memmap_path, mode="r")
        self.index_pairs = load_index_pairs(index_pairs=index_pairs, pairs_path=pairs_path)
        if len(self.index_pairs) == 0:
            raise ValueError("NavierStokesForecastingDataset requires at least one (item, t) pair.")
        first_item = int(self.index_pairs[0, 0])
        seq0 = self.mem[first_item]
        self.T = int(seq0.shape[0])
        self.H = int(seq0.shape[1])
        self.W = int(seq0.shape[2])

        self.sparsity = float(sparsity)
        self.mean = float(mean)
        self.std = float(std)
        self.clip_value = float(clip_value)
        self.mask_seed = int(mask_seed)
        self.normalize = bool(normalize)

    def __len__(self) -> int:
        return len(self.index_pairs)

    def __getitem__(self, idx: int):
        item_idx, t = map(int, self.index_pairs[idx])
        seq = self.mem[item_idx]
        x = np.asarray(seq[t], dtype=np.float32)
        y = np.asarray(seq[t + 1], dtype=np.float32)
        mask = make_fixed_mask(item_idx, t, self.T, self.sparsity, self.mask_seed, (self.H, self.W)).astype(np.bool_)

        if self.normalize:
            x_norm = (x - self.mean) / self.std
            y_norm = (y - self.mean) / self.std
        else:
            x_norm = x
            y_norm = y

        sample_id = compute_sample_id(item_idx, self.T, t)

        return (
            torch.from_numpy(x_norm[None, ...].astype(np.float32)),
            torch.from_numpy(y_norm[None, ...].astype(np.float32)),
            torch.from_numpy(mask),
            torch.tensor(sample_id, dtype=torch.long),
        )


In [ ]:
mem = RaggedMmap(MEMMAP_PATH, mode="r")
train_pairs, val_pairs, T, train_items, val_items = build_fixed_item_splits(
    mem,
    target_train_pairs=CONFIG["train_pairs"],
    val_pairs=CONFIG["val_pairs"],
    train_seed=CONFIG["gan"]["seed"],
    val_seed=VAL_SEED,
)

dataset_tag = dataset_tag_from_pairs(len(train_pairs))
sparsity_tag = f"{int(round(CONFIG['sparsity'] * 100))}pct"
STYLEGAN_RUN_ROOT = os.path.join("NavierStokesCheckpoints", "StyleGAN3", dataset_tag, sparsity_tag)
os.makedirs(STYLEGAN_RUN_ROOT, exist_ok=True)

stylegan_train_dataset = NavierStokesStyleGANDataset(
    memmap_path=MEMMAP_PATH,
    index_pairs=train_pairs,
    sparsity=CONFIG["sparsity"],
    mean=CONFIG["mean"],
    std=CONFIG["std"],
    clip_value=CONFIG["clip_value"],
    mask_seed=CONFIG["mask_seed"],
    name=f"NavierStokes-{dataset_tag}-{sparsity_tag}",
)

print(
    f"Train samples: {len(stylegan_train_dataset)} | Label dim: {stylegan_train_dataset.label_dim} | Resolution: {stylegan_train_dataset.resolution}"
)

val_dataset = NavierStokesForecastingDataset(
    memmap_path=MEMMAP_PATH,
    index_pairs=val_pairs,
    sparsity=CONFIG["sparsity"],
    mean=CONFIG["mean"],
    std=CONFIG["std"],
    clip_value=CONFIG["clip_value"],
    mask_seed=CONFIG["mask_seed"],
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["evaluation"]["batch_size"],
    shuffle=False,
    num_workers=CONFIG["evaluation"]["num_workers"],
)


In [ ]:
sample_img_uint8, sample_label = stylegan_train_dataset[0]
obs_sparse_scaled, obs_mask = parse_condition_vector(sample_label, stylegan_train_dataset.H, stylegan_train_dataset.W)
target_scaled = uint8_to_scaled(sample_img_uint8[0])
obs_field = scaled_to_physical(obs_sparse_scaled, CONFIG["clip_value"], CONFIG["mean"], CONFIG["std"])
target_field = scaled_to_physical(target_scaled, CONFIG["clip_value"], CONFIG["mean"], CONFIG["std"])

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
im0 = axes[0].imshow(obs_field, cmap="viridis")
axes[0].set_title("Masked input (physical units)")
axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(target_field, cmap="viridis")
axes[1].set_title("Forecast target t+1")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(obs_mask, cmap="gray")
axes[2].set_title("Visible mask")
axes[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
def build_stylegan3_training_config(run_root: str, run_suffix: str = "stylegan3") -> dnnlib.EasyDict:
    cfg = CONFIG["gan"]
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    run_dir = os.path.join(run_root, f"{timestamp}-{run_suffix}")
    os.makedirs(run_dir, exist_ok=False)

    pairs_path = os.path.join(run_dir, "train_pairs.npy")
    np.save(pairs_path, train_pairs)

    training_set_kwargs = dnnlib.EasyDict(
        class_name="__main__.NavierStokesStyleGANDataset",
        memmap_path=MEMMAP_PATH,
        pairs_path=pairs_path,
        sparsity=CONFIG["sparsity"],
        mean=CONFIG["mean"],
        std=CONFIG["std"],
        clip_value=CONFIG["clip_value"],
        mask_seed=CONFIG["mask_seed"],
        name=f"NavierStokes-{dataset_tag}-{sparsity_tag}",
        random_seed=cfg["seed"],
        xflip=False,
    )

    data_loader_kwargs = dnnlib.EasyDict(pin_memory=True, prefetch_factor=2, num_workers=cfg["num_workers"])

    batch_gpu = cfg["batch_gpu"] or cfg["batch_size"]

    G_kwargs = dnnlib.EasyDict(
        class_name="training.networks_stylegan3.Generator",
        z_dim=512,
        w_dim=512,
        mapping_kwargs=dnnlib.EasyDict(),
        channel_base=cfg["cbase"],
        channel_max=cfg["cmax"],
    )
    D_kwargs = dnnlib.EasyDict(
        class_name="training.networks_stylegan2.Discriminator",
        block_kwargs=dnnlib.EasyDict(freeze_layers=cfg["freezed"]),
        mapping_kwargs=dnnlib.EasyDict(),
        epilogue_kwargs=dnnlib.EasyDict(mbstd_group_size=cfg["mbstd"]),
        channel_base=cfg["cbase"],
        channel_max=cfg["cmax"],
    )

    if cfg["map_depth"] is not None:
        G_kwargs.mapping_kwargs.num_layers = cfg["map_depth"]
    else:
        G_kwargs.mapping_kwargs.num_layers = 2

    if cfg["cfg"] == "stylegan3-r":
        G_kwargs.conv_kernel = 1
        G_kwargs.channel_base *= 2
        G_kwargs.channel_max *= 2
        G_kwargs.use_radial_filters = True
        loss_kwargs = dnnlib.EasyDict(
            class_name="training.loss.StyleGAN2Loss",
            r1_gamma=cfg["gamma"],
            blur_init_sigma=10,
            blur_fade_kimg=cfg["batch_size"] * 200 / 32,
        )
    elif cfg["cfg"] == "stylegan3-t":
        G_kwargs.magnitude_ema_beta = 0.5 ** (cfg["batch_size"] / (20 * 1e3))
        loss_kwargs = dnnlib.EasyDict(class_name="training.loss.StyleGAN2Loss", r1_gamma=cfg["gamma"])
    else:
        raise ValueError("Set CONFIG['gan']['cfg'] to a StyleGAN3 variant for this baseline.")

    G_opt_kwargs = dnnlib.EasyDict(class_name="torch.optim.Adam", betas=[0, 0.99], eps=1e-8, lr=cfg["glr"] or 0.0025)
    D_opt_kwargs = dnnlib.EasyDict(class_name="torch.optim.Adam", betas=[0, 0.99], eps=1e-8, lr=cfg["dlr"])

    config = dnnlib.EasyDict(
        run_dir=run_dir,
        training_set_kwargs=training_set_kwargs,
        data_loader_kwargs=data_loader_kwargs,
        G_kwargs=G_kwargs,
        D_kwargs=D_kwargs,
        G_opt_kwargs=G_opt_kwargs,
        D_opt_kwargs=D_opt_kwargs,
        augment_kwargs=None,
        loss_kwargs=loss_kwargs,
        metrics=list(cfg["metrics"]),
        random_seed=cfg["seed"],
        num_gpus=1,
        batch_size=cfg["batch_size"],
        batch_gpu=batch_gpu,
        ema_kimg=cfg["batch_size"] * 10 / 32,
        ema_rampup=0.05,
        G_reg_interval=None,
        D_reg_interval=16,
        ada_interval=cfg["ada_interval"],
        total_kimg=cfg["total_kimg"],
        kimg_per_tick=cfg["tick_kimg"],
        image_snapshot_ticks=cfg["snapshot_ticks"],
        network_snapshot_ticks=cfg["snapshot_ticks"],
        augment_p=0.0,
        ada_kimg=500,
        progress_fn=None,
        cudnn_benchmark=True,
    )

    if cfg["augment"] != "noaug":
        config.augment_kwargs = dnnlib.EasyDict(
            class_name="training.augment.AugmentPipe",
            xflip=1,
            rotate90=1,
            xint=1,
            scale=1,
            rotate=1,
            aniso=1,
            xfrac=1,
            brightness=1,
            contrast=1,
            lumaflip=1,
            hue=1,
            saturation=1,
        )
        if cfg["augment"] == "ada":
            config.ada_target = cfg["ada_target"]
        elif cfg["augment"] == "fixed":
            config.augment_p = cfg["augment_p"]

    if cfg["resume_pkl"]:
        config.resume_pkl = cfg["resume_pkl"]
        config.ada_kimg = 100
        config.ema_rampup = None
        config.loss_kwargs.blur_init_sigma = 0

    return config


In [ ]:
def run_stylegan3_training(config: dnnlib.EasyDict) -> None:
    os.makedirs(config.run_dir, exist_ok=True)
    export_training_config(config, os.path.join(config.run_dir, "notebook_config.json"))
    dnnlib.util.Logger(file_name=os.path.join(config.run_dir, "log.txt"), file_mode="a", should_flush=True)
    print(f"Launching StyleGAN3 training in {config.run_dir}")
    print(f"Batch size: {config.batch_size} | Total kimg: {config.total_kimg} | gamma: {config.loss_kwargs.r1_gamma}")
    training_loop.training_loop(rank=0, **config)


In [ ]:
training_config = build_stylegan3_training_config(STYLEGAN_RUN_ROOT, run_suffix="stylegan3")
print(f"Run directory: {training_config.run_dir}")
print(f"Augmentation: {CONFIG['gan']['augment']} | Metrics: {training_config.metrics}")

# Uncomment the next line to launch full training.
# run_stylegan3_training(training_config)


In [ ]:
def find_latest_snapshot(run_dir: str) -> str:
    candidates = sorted(glob.glob(os.path.join(run_dir, "network-snapshot-*.pkl")))
    return candidates[-1] if candidates else None


def load_generator(snapshot_path: str, device: str) -> torch.nn.Module:
    with dnnlib.util.open_url(snapshot_path) as f:
        data = legacy.load_network_pkl(f)
    G_ema = data["G_ema"].to(device)
    G_ema.eval().requires_grad_(False)
    return G_ema


def evaluate_generator(
    G_ema: torch.nn.Module,
    data_loader: DataLoader,
    clip_value: float,
    mean: float,
    std: float,
    device: str,
    crps_K: int = 10,
    truncation_psi: float = 1.0,
) -> Dict[str, float]:
    mse_sum = 0.0
    mae_sum = 0.0
    crps_sum = 0.0
    mse_unnorm_sum = 0.0
    mae_unnorm_sum = 0.0
    crps_unnorm_sum = 0.0
    total = 0

    progress = tqdm(data_loader, desc="Evaluating StyleGAN3", leave=False)
    with torch.no_grad():
        for x_norm, y_norm, mask, _ in progress:
            x_norm = x_norm.to(device)
            y_norm = y_norm.to(device)
            mask = mask.to(device)

            cond = prepare_condition_tensor(x_norm, mask, clip_value)
            batch_size = x_norm.shape[0]

            z = torch.randn(crps_K * batch_size, G_ema.z_dim, device=device)
            c = cond.repeat(crps_K, 1).to(device)

            preds = G_ema(z, c, truncation_psi=truncation_psi, noise_mode="random")
            preds = preds.reshape(crps_K, batch_size, *preds.shape[1:])

            target_scaled = torch.clamp(y_norm, -clip_value, clip_value) / clip_value
            mean_pred = preds.mean(dim=0)

            mse_sum += F.mse_loss(mean_pred, target_scaled, reduction="sum").item()
            mae_sum += F.l1_loss(mean_pred, target_scaled, reduction="sum").item()

            crps_val = compute_crps(preds, target_scaled)
            crps_sum += crps_val.item() * batch_size

            preds_phys = scaled_to_physical_tensor(preds, clip_value, mean, std)
            mean_pred_phys = preds_phys.mean(dim=0)
            target_phys = scaled_to_physical_tensor(target_scaled, clip_value, mean, std)

            mse_unnorm_sum += F.mse_loss(mean_pred_phys, target_phys, reduction="sum").item()
            mae_unnorm_sum += F.l1_loss(mean_pred_phys, target_phys, reduction="sum").item()

            crps_phys = compute_crps(preds_phys, target_phys)
            crps_unnorm_sum += crps_phys.item() * batch_size

            total += batch_size

    metrics = {
        "mse": mse_sum / total,
        "rmse": math.sqrt(mse_sum / total),
        "mae": mae_sum / total,
        "crps": crps_sum / total,
        "mse_unnorm": mse_unnorm_sum / total,
        "rmse_unnorm": math.sqrt(mse_unnorm_sum / total),
        "mae_unnorm": mae_unnorm_sum / total,
        "crps_unnorm": crps_unnorm_sum / total,
    }
    return metrics


In [ ]:
latest_snapshot = find_latest_snapshot(training_config.run_dir)
print(f"Latest snapshot: {latest_snapshot}")

if latest_snapshot:
    G_ema = load_generator(latest_snapshot, CONFIG["device"])
    val_metrics = evaluate_generator(
        G_ema,
        val_loader,
        clip_value=CONFIG["clip_value"],
        mean=CONFIG["mean"],
        std=CONFIG["std"],
        device=CONFIG["device"],
        crps_K=CONFIG["evaluation"]["crps_k"],
        truncation_psi=CONFIG["gan"]["truncation_psi"],
    )
    print(json.dumps(val_metrics, indent=2))
else:
    print("Train the model to generate snapshots before running evaluation.")


In [ ]:
def visualize_forecasts(
    G_ema: torch.nn.Module,
    dataset: NavierStokesForecastingDataset,
    seeds=(0, 1, 2),
    clip_value: float = 5.0,
    mean: float = 0.0,
    std: float = 1.0,
    device: str = "cpu",
    truncation_psi: float = 1.0,
    max_examples: int = 2,
) -> None:
    G_ema = G_ema.to(device).eval()
    for idx in range(min(len(dataset), max_examples)):
        x_norm, y_norm, mask, _ = dataset[idx]
        x_norm = x_norm.unsqueeze(0).to(device)
        y_norm = y_norm.unsqueeze(0).to(device)
        mask = mask.unsqueeze(0).to(device)
        cond = prepare_condition_tensor(x_norm, mask, clip_value)

        x_sparse = torch.clamp(x_norm, -clip_value, clip_value) / clip_value
        x_sparse = x_sparse * mask.unsqueeze(1).to(x_sparse.dtype)
        x_field = scaled_to_physical_tensor(x_sparse, clip_value, mean, std).squeeze().cpu().numpy()
        y_field = scaled_to_physical_tensor(torch.clamp(y_norm, -clip_value, clip_value) / clip_value, clip_value, mean, std).squeeze().cpu().numpy()

        fig, axes = plt.subplots(1, len(seeds) + 2, figsize=(4 * (len(seeds) + 2), 4))
        axes[0].imshow(x_field, cmap="viridis")
        axes[0].set_title("Masked input")
        axes[0].axis("off")
        axes[1].imshow(y_field, cmap="viridis")
        axes[1].set_title("Target")
        axes[1].axis("off")

        for col, seed in enumerate(seeds, start=2):
            gen = torch.Generator(device=device)
            gen.manual_seed(seed)
            z = torch.randn(1, G_ema.z_dim, generator=gen, device=device)
            pred = G_ema(z, cond, truncation_psi=truncation_psi, noise_mode="random")
            pred_field = scaled_to_physical_tensor(pred, clip_value, mean, std).squeeze().cpu().numpy()
            axes[col].imshow(pred_field, cmap="viridis")
            axes[col].set_title(f"Seed {seed}")
            axes[col].axis("off")

        plt.tight_layout()
        plt.show()


In [ ]:
# Example usage after training:
# if latest_snapshot:
#     G_ema = load_generator(latest_snapshot, CONFIG["device"])
#     visualize_forecasts(
#         G_ema,
#         val_dataset,
#         seeds=(0, 1, 2),
#         clip_value=CONFIG["clip_value"],
#         mean=CONFIG["mean"],
#         std=CONFIG["std"],
#         device=CONFIG["device"],
#         truncation_psi=CONFIG["gan"]["truncation_psi"],
#     )
